In [ ]:
#######################################专用切割代码实现########################################
import re
import json
from typing import List, Dict, Tuple
import fitz  # PyMuPDF

class JinHuaReportChunker:
    """金花股份财报专用切割器"""
    
    def __init__(self):
        # 基于文档结构的切割配置
        self.section_patterns = {
            # 一级标题（目录中的主要章节）
            "level1": r'^(第[一二三四五六七八九十]+节|第一节|第二节|第三节|第四节|第五节|第六节|第七节|第八节|第九节|第十节)\s+',
            # 二级标题
            "level2": r'^[一二三四五六七八九十]+、',
            # 三级标题
            "level3": r'^\(\w+\)',
            # 表格标识
            "table": r'^单位：.*元.*币种：.*人民币'
        }
        
        # 关键数据区域（需要保留完整性的区域）
        self.critical_zones = [
            "近三年主要会计数据和财务指标",
            "分季度主要财务数据",
            "主营业务分行业、分产品、分地区",
            "产销量情况分析表",
            "研发投入情况",
            "主要销售客户及主要供应商情况"
        ]
    
    def extract_by_structure(self, pdf_path: str) -> List[Dict]:
        """按文档结构提取并切割"""
        doc = fitz.open(pdf_path)
        
        all_chunks = []
        current_section = {"title": "", "content": [], "page_start": 0}
        
        for page_num, page in enumerate(doc):
            text = page.get_text()
            lines = text.split('\n')
            
            for line in lines:
                line = line.strip()
                if not line:
                    continue
                
                # 检测新章节开始
                section_match = self._detect_section(line)
                if section_match:
                    # 保存上一个章节
                    if current_section["content"]:
                        chunk = self._create_chunk(current_section, page_num)
                        all_chunks.append(chunk)
                    
                    # 开始新章节
                    current_section = {
                        "title": section_match,
                        "content": [line],
                        "page_start": page_num
                    }
                else:
                    current_section["content"].append(line)
        
        # 保存最后一个章节
        if current_section["content"]:
            all_chunks.append(self._create_chunk(current_section, page_num))
        
        return all_chunks
    
    def _detect_section(self, line: str) -> str:
        """检测是否为章节标题"""
        for level, pattern in self.section_patterns.items():
            if re.match(pattern, line):
                return line
        return None
    
    def _create_chunk(self, section: Dict, page_num: int) -> Dict:
        """创建带元数据的文本块"""
        content = '\n'.join(section["content"])
        
        # 判断是否包含表格
        has_table = '│' in content or '┌' in content or '单位：' in content
        
        # 提取关键财务指标
        indicators = self._extract_indicators(content)
        
        return {
            "text": content,
            "metadata": {
                "title": section["title"],
                "page": page_num,
                "has_table": has_table,
                "indicators": indicators,
                "section_type": self._classify_section(section["title"])
            }
        }
    
    def _extract_indicators(self, text: str) -> List[str]:
        """提取文本中的财务指标"""
        indicator_patterns = [
            r'营业收入[\s:：]*[\d,\.]+',
            r'净利润[\s:：]*[\d,\.]+',
            r'归属于上市公司股东的净利润[\s:：]*[\d,\.]+',
            r'基本每股收益[\s:：]*[\d\.]+',
            r'加权平均净资产收益率[\s:：]*[\d\.]+%',
            r'经营活动产生的现金流量净额[\s:：]*[\d,\.]+'
        ]
        
        found = []
        for pattern in indicator_patterns:
            matches = re.findall(pattern, text)
            found.extend(matches)
        return found
    
    def _classify_section(self, title: str) -> str:
        """分类章节类型"""
        if '财务指标' in title or '会计数据' in title:
            return "financial_data"
        elif '经营情况' in title or '主营业务' in title:
            return "business_analysis"
        elif '资产负债表' in title or '利润表' in title or '现金流量表' in title:
            return "financial_statement"
        elif '研发' in title:
            return "rnd"
        elif '股东' in title:
            return "shareholder"
        else:
            return "general"
#################################表格专用处理###########################################
class TableProcessor:
    """财报表格专用处理器"""
    
    def extract_key_tables(self, text: str) -> List[Dict]:
        """提取财报中的关键表格"""
        tables = []
        
        # 识别并提取各个关键表格
        table_patterns = {
            "近三年主要会计数据": r'主要会计数据.*?(\n\|.*?\|.*?\n)+',
            "分季度数据": r'分季度.*?(\n\|.*?\|.*?\n)+',
            "主营业务分产品": r'主营业务分产品情况.*?(\n\|.*?\|.*?\n)+',
            "产销量情况": r'产销量情况分析表.*?(\n\|.*?\|.*?\n)+',
            "研发投入": r'研发投入情况.*?(\n\|.*?\|.*?\n)+'
        }
        
        for name, pattern in table_patterns.items():
            match = re.search(pattern, text, re.DOTALL)
            if match:
                table_text = match.group(0)
                tables.append({
                    "name": name,
                    "text": table_text,
                    "structured": self._parse_markdown_table(table_text)
                })
        
        return tables
    
    def _parse_markdown_table(self, table_text: str) -> List[Dict]:
        """解析Markdown格式表格"""
        lines = table_text.strip().split('\n')
        if len(lines) < 3:
            return []
        
        # 提取表头
        headers = [h.strip() for h in lines[0].split('|')[1:-1]]
        
        # 提取数据行
        data = []
        for line in lines[2:]:  # 跳过分隔行
            if '|' in line:
                values = [v.strip() for v in line.split('|')[1:-1]]
                if values:
                    row = dict(zip(headers, values))
                    data.append(row)
        
        return data       
##########################向量化与存储#######################################


In [ ]:
#一、文本切割（Chunking）策略
#1.1 针对财报和研报的专用切割策略
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    SentenceTransformersTokenTextSplitter
)
import re

class FinancialDocumentChunker:
    """财报/研报专用文本切割器"""
    
    def __init__(self):
        self.configs = {
            # 财报PDF切割配置
            "financial_report": {
                "chunk_size": 800,
                "chunk_overlap": 200,
                "separators": ["\n\n", "\n", "。", "；", "，", " ", ""],
                "keep_separator": False
            },
            # 研报切割配置（更小的块便于检索）
            "research_report": {
                "chunk_size": 500,
                "chunk_overlap": 150,
                "separators": ["\n\n", "\n", "。", "；", " ", ""]
            },
            # 表格数据特殊处理
            "table_data": {
                "chunk_size": 400,
                "chunk_overlap": 80,
                "separators": ["\n", "\t", "|"]
            }
        }
        
        # 自定义分割正则（保留关键结构）
        self.structure_patterns = {
            "section_title": r"^(第[一二三四五六七八九十]+章|【.*?】|#{1,3}\s+.*)$",
            "table_start": r"^\s*┌|---\|",
            "table_end": r"^\s*└|---\|",
            "financial_indicator": r"(营业收入|净利润|毛利率|ROE|资产负债率)[\s:：]*[\d,\.]+"
        }
    
    def chunk_by_semantic_boundary(self, text, doc_type="research_report"):
        """基于语义边界的智能切割"""
        config = self.configs[doc_type]
        
        # 1. 先用标题切分（保持章节完整性）
        sections = self._split_by_headings(text)
        
        chunks = []
        for section in sections:
            # 2. 对每个章节递归切割
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=config["chunk_size"],
                chunk_overlap=config["chunk_overlap"],
                separators=config["separators"],
                keep_separator=config.get("keep_separator", False)
            )
            section_chunks = splitter.split_text(section)
            
            # 3. 添加元数据标记
            for chunk in section_chunks:
                chunks.append({
                    "text": chunk,
                    "metadata": self._extract_metadata(chunk, doc_type)
                })
        
        return chunks
    
    def _split_by_headings(self, text):
        """按标题层级切分"""
        # 匹配中英文标题
        heading_pattern = r'(?:^|\n)(#{1,3}\s+.+?|\d+\.\d+\s+.+?|[一二三四五]+、.+?)(?=\n|$)'
        
        # 简单实现：按段落分割
        paragraphs = re.split(r'\n\s*\n', text)
        
        sections = []
        current_section = []
        
        for para in paragraphs:
            if re.match(self.structure_patterns["section_title"], para.strip()):
                # 遇到新标题，保存上一个section
                if current_section:
                    sections.append('\n\n'.join(current_section))
                current_section = [para]
            else:
                current_section.append(para)
        
        if current_section:
            sections.append('\n\n'.join(current_section))
        
        return sections if sections else [text]
    
    def _extract_metadata(self, chunk, doc_type):
        """提取chunk元数据"""
        metadata = {
            "doc_type": doc_type,
            "has_table": bool(re.search(r'┌|─|─|│|表\d+', chunk)),
            "has_number": bool(re.search(r'\d+\.?\d*%?', chunk)),
            "indicators": re.findall(self.structure_patterns["financial_indicator"], chunk)
        }
        return metadata
    
    def chunk_with_overlap_strategy(self, text, doc_type="research_report"):
        """滑动窗口重叠切割（保证上下文连贯）"""
        config = self.configs[doc_type]
        
        # 使用句子边界分词器
        splitter = SentenceTransformersTokenTextSplitter(
            chunk_size=config["chunk_size"],
            chunk_overlap=config["chunk_overlap"],
            model_name="BAAI/bge-large-zh-v1.5"
        )
        
        chunks = splitter.split_text(text)
        
        # 添加滑动窗口标记
        result = []
        for i, chunk in enumerate(chunks):
            result.append({
                "text": chunk,
                "chunk_id": i,
                "prev_context": chunks[i-1][-200:] if i > 0 else "",
                "next_context": chunks[i+1][:200] if i < len(chunks)-1 else ""
            })
        
        return result
    
#1.2 针对财报表格的专用切割
import pandas as pd
import fitz  # PyMuPDF

class TableAwareChunker:
    """表格感知的切割器"""
    
    def extract_tables_from_pdf(self, pdf_path):
        """从PDF中提取表格"""
        tables = []
        doc = fitz.open(pdf_path)
        
        for page_num, page in enumerate(doc):
            # 提取页面中的表格
            page_tables = page.find_tables()
            for table in page_tables:
                df = table.to_pandas()
                tables.append({
                    "page": page_num,
                    "data": df,
                    "text_representation": self._table_to_markdown(df),
                    "metadata": {
                        "indicators": list(df.columns),
                        "periods": self._extract_periods(df)
                    }
                })
        
        return tables
    
    def _table_to_markdown(self, df):
        """将表格转为Markdown格式（保留语义）"""
        # 限制表格大小，避免过大
        if len(df) > 20:
            df = df.head(20)
        
        return df.to_markdown()
    
    def chunk_table(self, df, max_rows=10):
        """将大表格按行切割"""
        chunks = []
        for i in range(0, len(df), max_rows):
            chunk_df = df.iloc[i:i+max_rows]
            chunks.append({
                "text": chunk_df.to_markdown(),
                "rows_range": f"{i}-{i+max_rows}",
                "indicators": list(chunk_df.columns)
            })
        return chunks
#二、向量化（Embedding）策略
#2.1 多模型Embedding配置
from sentence_transformers import SentenceTransformer
from langchain.embeddings import HuggingFaceEmbeddings
import numpy as np
from typing import List, Dict

class FinancialEmbedding:
    """财报领域专用向量化器"""
    
    def __init__(self):
        # 中文金融领域最佳模型
        self.models = {
            "primary": "BAAI/bge-large-zh-v1.5",      # 主模型，维度1024
            "secondary": "shibing624/text2vec-base-chinese",  # 备选
            "finance_specific": "moka-ai/m3e-large"   # 金融领域微调模型
        }
        
        # 加载主模型
        self.encoder = SentenceTransformer(self.models["primary"])
        self.encoder.max_seq_length = 512  # 限制序列长度
        
        # 批量处理配置
        self.batch_size = 32
        
    def encode_single(self, text: str) -> np.ndarray:
        """单条文本向量化"""
        return self.encoder.encode(text, normalize_embeddings=True)
    
    def encode_batch(self, texts: List[str], show_progress=True) -> np.ndarray:
        """批量向量化（高效）"""
        embeddings = self.encoder.encode(
            texts,
            batch_size=self.batch_size,
            normalize_embeddings=True,
            show_progress_bar=show_progress
        )
        return embeddings
    
    def encode_with_metadata(self, chunks: List[Dict]) -> List[Dict]:
        """带元数据的向量化"""
        texts = [chunk["text"] for chunk in chunks]
        embeddings = self.encode_batch(texts)
        
        for i, chunk in enumerate(chunks):
            chunk["embedding"] = embeddings[i].tolist()
        
        return chunks
#2.2 增强向量化（添加金融领域知识）
class EnhancedFinancialEmbedding:
    """增强的金融向量化（融合领域知识）"""
    
    def __init__(self, base_encoder):
        self.base_encoder = base_encoder
        
        # 金融领域关键词权重
        self.finance_keywords = {
            "high_weight": ["营业收入", "净利润", "毛利率", "ROE", "资产负债率", 
                           "现金流量", "每股收益", "估值"],
            "medium_weight": ["增长", "下降", "同比", "环比", "预测", "评级"],
            "low_weight": ["公司", "行业", "市场", "报告"]
        }
    
    def encode_with_keyword_boost(self, text: str) -> np.ndarray:
        """关键词加权向量化"""
        base_embedding = self.base_encoder.encode_single(text)
        
        # 计算关键词权重
        boost_factor = 1.0
        for keyword in self.finance_keywords["high_weight"]:
            if keyword in text:
                boost_factor *= 1.2
        
        return base_embedding * boost_factor
    
    def encode_hybrid(self, text: str) -> np.ndarray:
        """混合向量化（稠密+稀疏）"""
        # 稠密向量
        dense_vec = self.base_encoder.encode_single(text)
        
        # 稀疏向量（关键词匹配）
        sparse_vec = self._build_sparse_vector(text)
        
        # 融合
        hybrid_vec = np.concatenate([dense_vec, sparse_vec])
        return hybrid_vec
    
    def _build_sparse_vector(self, text: str, dim=256) -> np.ndarray:
        """构建稀疏向量（基于关键词）"""
        sparse = np.zeros(dim)
        for i, keyword in enumerate(self.finance_keywords["high_weight"]):
            if keyword in text:
                sparse[i % dim] += text.count(keyword)
        # L2归一化
        norm = np.linalg.norm(sparse)
        if norm > 0:
            sparse = sparse / norm
        return sparse
#三、完整处理流程
import os
from typing import List, Dict
import chromadb
from chromadb.config import Settings

class RAGPipeline:
    """完整的RAG处理流水线"""
    
    def __init__(self, persist_directory="./chroma_db"):
        # 初始化切割器
        self.chunker = FinancialDocumentChunker()
        self.table_chunker = TableAwareChunker()
        
        # 初始化向量化器
        self.embedder = FinancialEmbedding()
        self.enhanced_embedder = EnhancedFinancialEmbedding(self.embedder)
        
        # 初始化向量数据库
        self.client = chromadb.Client(Settings(
            chroma_db_impl="duckdb+parquet",
            persist_directory=persist_directory
        ))
        
        # 创建集合（按文档类型分）
        self.collections = {
            "financial_reports": self.client.create_collection(
                name="financial_reports",
                metadata={"hnsw:space": "cosine"}
            ),
            "research_reports": self.client.create_collection(
                name="research_reports",
                metadata={"hnsw:space": "cosine"}
            )
        }
    
    def process_financial_report(self, pdf_path: str, company_name: str):
        """处理财报PDF"""
        print(f"处理财报: {pdf_path}")
        
        # 1. 提取文本
        text = self._extract_text_from_pdf(pdf_path)
        
        # 2. 切割文本
        chunks = self.chunker.chunk_by_semantic_boundary(
            text, doc_type="financial_report"
        )
        
        # 3. 提取表格并单独处理
        tables = self.table_chunker.extract_tables_from_pdf(pdf_path)
        for table in tables:
            chunks.append({
                "text": table["text_representation"],
                "metadata": {
                    "type": "table",
                    "page": table["page"],
                    **table["metadata"]
                }
            })
        
        # 4. 向量化
        for chunk in chunks:
            chunk["embedding"] = self.embedder.encode_single(chunk["text"])
        
        # 5. 存入向量数据库
        self._store_to_db(chunks, "financial_reports", company_name)
        
        return len(chunks)
    
    def process_research_report(self, pdf_path: str, report_metadata: Dict):
        """处理研报PDF"""
        print(f"处理研报: {pdf_path}")
        
        text = self._extract_text_from_pdf(pdf_path)
        
        # 研报使用更小的chunk
        chunks = self.chunker.chunk_with_overlap_strategy(
            text, doc_type="research_report"
        )
        
        # 添加研报元数据
        for chunk in chunks:
            chunk["metadata"] = {
                "report_type": report_metadata.get("type", "individual"),
                "company": report_metadata.get("company", ""),
                "industry": report_metadata.get("industry", ""),
                "date": report_metadata.get("date", ""),
                "title": report_metadata.get("title", "")
            }
        
        # 使用增强向量化
        for chunk in chunks:
            chunk["embedding"] = self.enhanced_embedder.encode_hybrid(chunk["text"])
        
        self._store_to_db(chunks, "research_reports", report_metadata.get("company", ""))
        
        return len(chunks)
    
    def _extract_text_from_pdf(self, pdf_path: str) -> str:
        """从PDF提取文本"""
        import fitz
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    
    def _store_to_db(self, chunks: List[Dict], collection_name: str, source: str):
        """存储到向量数据库"""
        collection = self.collections[collection_name]
        
        # 批量存储
        ids = [f"{source}_{i}" for i in range(len(chunks))]
        embeddings = [chunk["embedding"] for chunk in chunks]
        documents = [chunk["text"] for chunk in chunks]
        metadatas = [chunk.get("metadata", {}) for chunk in chunks]
        
        # 分批存储（避免内存溢出）
        batch_size = 100
        for i in range(0, len(ids), batch_size):
            collection.add(
                ids=ids[i:i+batch_size],
                embeddings=embeddings[i:i+batch_size],
                documents=documents[i:i+batch_size],
                metadatas=metadatas[i:i+batch_size]
            )
    
    def search(self, query: str, collection_name: str, top_k=5):
        """检索相关文档"""
        collection = self.collections[collection_name]
        query_embedding = self.embedder.encode_single(query)
        
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )
        
        return results